In [2]:
# Cell 1: Setup - Imports and Paths
import pandas as pd
import numpy as np
import os
import time
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    matthews_corrcoef
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, RepeatVector, TimeDistributed, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("✅ Setup complete.")

# --- Define Your Specific Paths ---
BASE_PATH = '/Users/hafizmohammedaahil/Documents/C3I-Code/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/'
DATA_PATH = os.path.join(BASE_PATH, 'data')
FEATURES_PATH = os.path.join(BASE_PATH, 'features') # Note: This path is no longer needed for intermediate files
MODEL_PATH = os.path.join(BASE_PATH, 'models')
RESULTS_PATH = os.path.join(BASE_PATH, 'results')

for path in [DATA_PATH, MODEL_PATH, RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

# --- Set Seed for Reproducibility ---
np.random.seed(42)

✅ Setup complete.


In [2]:
# Cell 2: Unified Data Loading and Preparation

def load_all_raw_data(data_path):
    """
    Loads, standardizes, and combines all raw data sources.
    """
    # --- Load Baltic Data ---
    print("Loading Baltic data...")
    baltic_path = os.path.join(data_path, 'datasets/training_set.csv')
    df_baltic = pd.read_csv(baltic_path, low_memory=False)
    # Standardize column names
    df_baltic.rename(columns={'lat': 'latitude', 'lon': 'longitude', 'timestamp': 'msgtime'}, inplace=True)
    df_baltic['source'] = 'baltic'
    print(f"Loaded {len(df_baltic)} rows from Baltic Sea.")

    # --- Load BarentsWatch (historical_polled) Data ---
    print("Loading BarentsWatch data...")
    barents_path = os.path.join(data_path, 'historical_polled.csv')
    df_barents = pd.read_csv(barents_path, low_memory=False)
    # Standardize column names
    df_barents.rename(columns={'speedOverGround': 'speed', 'courseOverGround': 'course'}, inplace=True)
    df_barents['source'] = 'barents'
    print(f"Loaded {len(df_barents)} rows from BarentsWatch.")

    # --- Combine and Select Common Columns ---
    # Ensure both dataframes have the same core columns before combining
    common_cols = ['mmsi', 'latitude', 'longitude', 'speed', 'course', 'msgtime', 'heading', 'source']
    
    # Use 'course' as a fallback for 'heading' if it's missing
    for df in [df_baltic, df_barents]:
        if 'heading' not in df.columns:
            print(f"Warning: 'heading' missing in {df['source'].iloc[0]} data. Using 'course' as fallback.")
            df['heading'] = df['course']
            
    df_master = pd.concat([
        df_baltic[common_cols], 
        df_barents[common_cols]
    ], ignore_index=True)
    
    return df_master

# 1. Load and Unify All Data
df_master = load_all_raw_data(DATA_PATH)
print(f"Loaded a total of {len(df_master)} data points from {df_master['source'].nunique()} sources.")

# 2. Clean the Master DataFrame
df_master.dropna(subset=['latitude', 'longitude', 'speed', 'course', 'heading'], inplace=True)
df_master.drop_duplicates(subset=['mmsi', 'msgtime'], inplace=True)

# Use robust percentile clipping
features_to_clip = ['latitude', 'longitude', 'speed', 'course', 'heading']
for col in features_to_clip:
    lower, upper = df_master[col].quantile(0.01), df_master[col].quantile(0.99)
    df_master[col] = df_master[col].clip(lower, upper)

print("Master DataFrame cleaned.")

# 3. Generate Synthetic Anomalies
df_master['label'] = 0
anomaly_indices = np.random.choice(df_master.index, size=int(0.1 * len(df_master)), replace=False)
anomaly_patterns = np.random.choice(['u_turn', 'loitering', 'speed_spike', 'pos_jump'], size=len(anomaly_indices))

for idx, pattern in zip(anomaly_indices, anomaly_patterns):
    if idx in df_master.index:
        if pattern == 'u_turn':
            df_master.loc[idx, 'course'] = (df_master.loc[idx, 'course'] + 180) % 360
            df_master.loc[idx, 'heading'] = (df_master.loc[idx, 'heading'] + 180) % 360
        elif pattern == 'loitering':
            df_master.loc[idx, 'speed'] = np.random.uniform(0, 0.5)
        elif pattern == 'speed_spike':
            df_master.loc[idx, 'speed'] *= np.random.uniform(2, 3)
        elif pattern == 'pos_jump':
            df_master.loc[idx, 'latitude'] += np.random.uniform(0.5, 1.0)
        
        df_master.loc[idx, 'label'] = 1

print(f"Generated {len(anomaly_indices)} anomalies in the master DataFrame.")
print(f"Final prepared DataFrame shape: {df_master.shape}")

Loading Baltic data...
Loaded 14185046 rows from Baltic Sea.
Loading BarentsWatch data...
Loaded 8726 rows from BarentsWatch.
Loaded a total of 14193772 data points from 2 sources.
Master DataFrame cleaned.
Generated 1376690 anomalies in the master DataFrame.
Final prepared DataFrame shape: (13766906, 9)


In [3]:
# Cell 3: Vessel-Level Split and Sequence Generation

# 1. Perform Vessel-Level Split on the clean Master DataFrame
unique_vessels = df_master['mmsi'].unique()
train_vessels, temp_vessels = train_test_split(unique_vessels, test_size=0.4, random_state=42)
val_vessels, test_vessels = train_test_split(temp_vessels, test_size=0.5, random_state=42)

df_train = df_master[df_master['mmsi'].isin(train_vessels)]
df_val = df_master[df_master['mmsi'].isin(val_vessels)]
df_test = df_master[df_master['mmsi'].isin(test_vessels)]

print(f"Data split by vessel: Train ({len(train_vessels)}), Val ({len(val_vessels)}), Test ({len(test_vessels)})")
print(f"Data points: Train ({len(df_train)}), Validation ({len(df_val)}), Test ({len(df_test)})")

# 2. Define a Single, Clean Sequence Builder Function
def create_sequences(df, features, seq_length=10):
    sequences, labels = [], []
    for mmsi, group in df.groupby('mmsi'):
        if len(group) < seq_length:
            continue

        feature_data = group[features].values
        label_data = group['label'].values

        for i in range(len(group) - seq_length + 1):
            seq = feature_data[i:i + seq_length]
            # Label sequence as anomalous if any point within it is an anomaly
            label = 1 if np.any(label_data[i:i + seq_length]) else 0

            # Contextual filter for stationary vessels
            if np.var(seq, axis=0).sum() == 0 and np.mean(seq[:, 2]) > 1:
                continue

            sequences.append(seq)
            labels.append(label)

    return np.array(sequences), np.array(labels)

# 3. Create Final Sequence Arrays from the Split DataFrames
features = ['latitude', 'longitude', 'speed', 'course', 'heading']
X_train, y_train = create_sequences(df_train, features)
X_val, y_val = create_sequences(df_val, features)
X_test, y_test = create_sequences(df_test, features)

print(f"\nFinal Shapes:")
print(f"Train:      X={X_train.shape}, y={y_train.shape}")
print(f"Validation: X={X_val.shape}, y={y_val.shape}")
print(f"Test:       X={X_test.shape}, y={y_test.shape}")

Data split by vessel: Train (1125), Val (375), Test (375)
Data points: Train (7942597), Validation (2912571), Test (2911738)

Final Shapes:
Train:      X=(7932556, 10, 5), y=(7932556,)
Validation: X=(2909225, 10, 5), y=(2909225,)
Test:       X=(2908397, 10, 5), y=(2908397,)


In [4]:
# Cell 4: Correct Data Scaling

# Isolate the normal data from the training set for fitting the scaler
X_train_normal = X_train[y_train == 0]

# Reshape data for scaling (3D -> 2D)
# Note: This might be memory intensive. If it crashes, you may need to process in chunks.
X_train_normal_reshaped = X_train_normal.reshape(-1, X_train_normal.shape[-1])

# Fit the scaler ONLY on the normal training data
scaler = StandardScaler()
scaler.fit(X_train_normal_reshaped)
print("✅ Scaler fitted on normal training data.")

# Save the scaler artifact to the MODEL_PATH
joblib.dump(scaler, os.path.join(MODEL_PATH, 'final_data_scaler.pkl'))

# Helper function to apply the fitted scaler
def scale_data(data, scaler_obj):
    # Reshape, scale, and reshape back
    reshaped = data.reshape(-1, data.shape[-1])
    scaled = scaler_obj.transform(reshaped)
    return scaled.reshape(data.shape)

# Apply the same scaler to all datasets
X_train_scaled = scale_data(X_train, scaler)
X_val_scaled = scale_data(X_val, scaler)
X_test_scaled = scale_data(X_test, scaler)

print("All datasets scaled successfully.")

✅ Scaler fitted on normal training data.
All datasets scaled successfully.


In [2]:
# Cell 5: Autoencoder Model Training

# Isolate the scaled normal training data for training the autoencoder
X_train_normal_scaled = X_train_scaled[y_train == 0]

# --- Optional but Recommended: Create a "denoising" version for more robust training ---
noise_factor = 0.05
X_train_noisy = X_train_normal_scaled + noise_factor * np.random.normal(size=X_train_normal_scaled.shape)

# Get dynamic shapes for the model
seq_length = X_train_normal_scaled.shape[1]
n_features = X_train_normal_scaled.shape[2]

# Define a clean and robust Bidirectional GRU Autoencoder
inputs = Input(shape=(seq_length, n_features))
encoder = Bidirectional(GRU(64, activation='relu', return_sequences=False))(inputs)
repeater = RepeatVector(seq_length)(encoder)
decoder = Bidirectional(GRU(64, activation='relu', return_sequences=True))(repeater)
outputs = TimeDistributed(Dense(n_features, activation='linear'))(decoder)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
model.summary()

# Define callbacks for efficient training
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
]

print("\n🚀 Training autoencoder on normal data only...")
history = model.fit(
    X_train_noisy,                # Use noisy data as input
    X_train_normal_scaled,        # Ask model to reconstruct the clean version
    epochs=150,
    batch_size=256,
    validation_data=(X_val_scaled, X_val_scaled), # Monitor against the full validation set
    callbacks=callbacks,
    verbose=1
)

model.save(os.path.join(MODEL_PATH, 'final_autoencoder_model.keras'))
print("🎉 Final model trained and saved.")

NameError: name 'X_train_scaled' is not defined

In [3]:
model.save(os.path.join(MODEL_PATH,'final_autoencoder_model.keras'))
print("Best Model (From Epoch 63) saved successfully!!!!!!")

NameError: name 'model' is not defined